# dark-ipfs notebook de prueba

Este notebook valida el cluster local de `dark-ipfs`:
1. Health de IPFS API y Cluster API
2. Carga de contenido y obtencion de CID
3. Pin via IPFS Cluster
4. Verificacion de lectura por gateway
5. Cleanup opcional (unpin)

Antes de ejecutar, levantar servicios:
```bash
cd /Users/lmatas/source/dark-developer/components/blockchain/dark-ipfs
make up
```

In [ ]:
import os
import time
import uuid
import requests

IPFS_API = os.getenv("IPFS_API", "http://localhost:5001")
GATEWAY = os.getenv("IPFS_GATEWAY", "http://localhost:38080")
CLUSTER_API = os.getenv("CLUSTER_API", "http://localhost:9094")
REPLICATION_MIN = int(os.getenv("CLUSTER_REPLICATION_MIN", "2"))
REPLICATION_MAX = int(os.getenv("CLUSTER_REPLICATION_MAX", "3"))

session = requests.Session()

print("IPFS_API:", IPFS_API)
print("GATEWAY:", GATEWAY)
print("CLUSTER_API:", CLUSTER_API)
print("REPLICATION:", REPLICATION_MIN, "-", REPLICATION_MAX)

In [ ]:
def check_ipfs_ready() -> dict:
    r = session.post(f"{IPFS_API}/api/v0/id", timeout=10)
    r.raise_for_status()
    return r.json()


def check_cluster_ready() -> dict:
    r = session.get(f"{CLUSTER_API}/id", timeout=10)
    r.raise_for_status()
    return r.json()


def wait_for_pin(cid: str, timeout: int = 60) -> dict:
    start = time.time()
    last = None
    while time.time() - start < timeout:
        r = session.get(f"{CLUSTER_API}/pins/{cid}", timeout=10)
        if r.status_code == 200:
            data = r.json()
            allocations = data.get("allocations", [])
            if len(allocations) >= REPLICATION_MIN:
                return data
            last = data
        time.sleep(2)
    raise TimeoutError(f"No se alcanzo replication min para {cid}. Ultimo estado: {last}")

In [ ]:
ipfs_id = check_ipfs_ready()
cluster_id = check_cluster_ready()

print("IPFS peer id:", ipfs_id.get("ID"))
print("Cluster peer id:", cluster_id.get("id"))

In [ ]:
payload = f"dark-ipfs-notebook-test-{uuid.uuid4()}"
files = {"file": ("payload.txt", payload.encode("utf-8"), "text/plain")}

add_resp = session.post(
    f"{IPFS_API}/api/v0/add",
    params={"cid-version": 1},
    files=files,
    timeout=20,
)
add_resp.raise_for_status()

cid = add_resp.json()["Hash"]
print("CID:", cid)
print("Payload:", payload)

In [ ]:
pin_resp = session.post(
    f"{CLUSTER_API}/pins/{cid}",
    params={
        "replication-min": REPLICATION_MIN,
        "replication-max": REPLICATION_MAX,
    },
    timeout=20,
)

# 200/202: pin aceptado, 409: ya existia
if pin_resp.status_code not in (200, 202, 409):
    raise RuntimeError(f"Error pinning CID {cid}: {pin_resp.status_code} - {pin_resp.text}")

pin_state = wait_for_pin(cid)
allocations = pin_state.get("allocations", [])

print("Pin response status:", pin_resp.status_code)
print("Allocations:", allocations)
print("Replication min satisfied:", len(allocations) >= REPLICATION_MIN)

In [ ]:
gateway_resp = session.get(
    f"{GATEWAY}/ipfs/{cid}",
    timeout=20,
    allow_redirects=False,
)

if gateway_resp.status_code in (301, 302, 303, 307, 308):
    redirect_to = gateway_resp.headers.get("Location", "")
    print("Gateway redirect detectado:", redirect_to)
    # Fallback robusto para entornos donde *.ipfs.localhost no resuelve DNS
    api_cat = session.post(
        f"{IPFS_API}/api/v0/cat",
        params={"arg": cid},
        timeout=20,
    )
    api_cat.raise_for_status()
    retrieved = api_cat.text
else:
    gateway_resp.raise_for_status()
    retrieved = gateway_resp.text

assert retrieved == payload, f"Contenido distinto. esperado={payload} obtenido={retrieved}"
print("OK: contenido recuperado coincide con payload")

In [ ]:
# Cleanup opcional
UNPIN_AT_END = False

if UNPIN_AT_END:
    unpin_resp = session.delete(f"{CLUSTER_API}/pins/{cid}", timeout=20)
    if unpin_resp.status_code not in (200, 202):
        raise RuntimeError(f"Error unpinning CID {cid}: {unpin_resp.status_code} - {unpin_resp.text}")
    print("CID unpinned:", cid)
else:
    print("Cleanup omitido. CID queda pinneado:", cid)